# Low-Resource Neural Machine Translation: English → Malayalam (Part 2 — Domain Adaptation)

This notebook continues from Part 1 (`low_resource_mt_malayalam.ipynb`), which fine-tuned `Helsinki-NLP/opus-mt-en-dra` on the AI4Bharat BPCC general-domain corpus (BLEU 15.48).

Here we adapt that baseline to technical domain text using the SPRINGLab Shiksha corpus — parallel English–Malayalam transcriptions of NPTEL university lectures — and evaluate on held-out technical data and out-of-domain ML arXiv abstracts.

## Results

| Model | Training Data | Evaluated On | BLEU | 1-gram | 2-gram | 3-gram | 4-gram | BP |
|-------|--------------|-------------|------|--------|--------|--------|--------|----|
| Baseline | BPCC (general domain) | BPCC val | 15.48 | 47.6 | 20.8 | 10.4 | 5.6 | 0.999 |
| Adapted | BPCC + Shiksha (technical) | Shiksha val | 32.61 | 66.6 | 40.9 | 26.2 | 17.4 | 0.976 |

The two models are evaluated on different validation sets — BPCC val (general domain) and Shiksha val (technical domain) respectively. The scores are not directly comparable as absolute numbers. The n-gram progression tells the real story: the adapted model improves dramatically on longer phrases (4-gram: 5.6 → 17.4), reflecting genuine gains in fluency and domain vocabulary rather than just unigram recall.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q sacrebleu datasets transformers sentencepiece

In [ ]:
import os
import re
import json
import random
import numpy as np
import torch

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. Load Baseline Model

Loading the BPCC-trained checkpoint saved in Part 1. We continue fine-tuning from this checkpoint rather than from the original pretrained weights — the model already has general Malayalam fluency, we are now adapting its vocabulary and register to technical domain text.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

model     = MarianMTModel.from_pretrained("/content/drive/MyDrive/marian_bpcc_final")
tokeniser = MarianTokenizer.from_pretrained("/content/drive/MyDrive/marian_bpcc_final")
model     = model.to(device)

print(f"Loaded. Vocab size: {tokeniser.vocab_size:,}")

## 3. Shiksha Dataset

[SPRINGLab Shiksha](https://huggingface.co/datasets/SPRINGLab/shiksha) is a parallel corpus mined from human-translated transcriptions of NPTEL video lectures covering engineering, mathematics, and sciences. It covers 8 Indian languages.

Language codes in this dataset are integers: English = 0, Malayalam = 5. We filter to English→Malayalam pairs and apply a quality score threshold of 0.85 to retain only high-confidence translations.

| Split | Pairs |
|-------|-------|
| Raw English→Malayalam | 135,080 |
| After quality filter (score > 0.85) | 54,395 |
| Train (80%) | 43,516 |
| Validation (20%) | 10,879 |

In [ ]:
from datasets import load_dataset

shiksha = load_dataset("SPRINGLab/shiksha", trust_remote_code=True)
print(shiksha)

In [ ]:
# Language codes: 0 = English, 5 = Malayalam
mal_data = shiksha['train'].filter(
    lambda x: x['src_lang'] == 0 and x['tgt_lang'] == 5 and x['score'] > 0.85
)
print(f"After quality filter: {len(mal_data):,}")
print(mal_data[0])

After quality filter: 54,395


In [ ]:
shiksha_split = mal_data.train_test_split(test_size=0.2, seed=SEED)
shiksha_train = shiksha_split["train"]
shiksha_val   = shiksha_split["test"]

print(f"Train: {len(shiksha_train):,}  |  Val: {len(shiksha_val):,}")

Train: 43,516  |  Val: 10,879


## 4. Preprocessing and Tokenisation

Shiksha uses `src_text`/`tgt_text` field names (vs BPCC's `src`/`tgt`). The `>>mal<<` language tag is still required — `opus-mt-en-dra` is a multilingual model and needs the tag to select Malayalam at generation time regardless of which dataset the data comes from.

**Why no padding here:** Padding is deferred to the data collator, which pads dynamically per batch to the longest sequence in that batch. This is more efficient than padding to a global maximum — shorter batches stay short.

In [ ]:
MAX_LEN = 256

def preprocess_batch(batch):
    src = [f">>mal<< {s}" for s in batch["src_text"]]
    tgt = batch["tgt_text"]

    model_inputs = tokeniser(
        src, max_length=MAX_LEN, truncation=True, padding=False,
    )
    labels = tokeniser(
        text_target=tgt, max_length=MAX_LEN, truncation=True, padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenised_shiksha = {}
for name, data in [("train", shiksha_train), ("validation", shiksha_val)]:
    tokenised_shiksha[name] = data.map(
        preprocess_batch,
        batched=True,
        remove_columns=data.column_names,
    )
    print(f"{name}: {len(tokenised_shiksha[name]):,} tokenised")

## 5. Domain Adaptation Fine-Tuning

We continue training from the BPCC checkpoint at a reduced learning rate (`2e-5` vs `5e-5` in Part 1). The lower rate is critical when continuing from a pretrained checkpoint — a higher rate would cause large weight updates that destabilise what the model already learned, a phenomenon called catastrophic forgetting.

Shorter warmup (200 steps vs 500) because the model is already warm — it does not need a long ramp-up from near-zero gradients.

The data collator handles dynamic padding per batch. `pad_to_multiple_of=8` aligns tensor dimensions to multiples of 8 for Tensor Core efficiency on T4 GPUs. Padding positions in labels are masked with `-100` so the cross-entropy loss ignores them — the model is only penalised for tokens it was actually supposed to predict.

BLEU is computed at the end of each epoch via beam search decoding (`num_beams=4`) on the Shiksha validation set.

In [ ]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from sacrebleu.metrics import BLEU

bleu_metric = BLEU()

data_collator = DataCollatorForSeq2Seq(
    tokeniser,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = np.where(labels != -100, labels, tokeniser.pad_token_id)
    decoded_preds  = tokeniser.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokeniser.batch_decode(labels, skip_special_tokens=True)
    result = bleu_metric.corpus_score(decoded_preds, [decoded_labels])
    return {"bleu": round(result.score, 2)}

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/marian_shiksha",
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=2e-5,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=256,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    logging_steps=100,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenised_shiksha["train"],
    eval_dataset=tokenised_shiksha["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

model.save_pretrained("/content/drive/MyDrive/marian_shiksha_final")
tokeniser.save_pretrained("/content/drive/MyDrive/marian_shiksha_final")
print("Saved.")

Epoch	Training Loss	Validation Loss	BLEU
1	1.476989	1.256926	28.93
2	1.246602	1.153274	31.18
3	1.145933	1.117021	32.24
4	1.096816	1.105637	32.65
Saved.


## 6. Evaluation

### 6.1 Shiksha Validation BLEU

Held-out Shiksha validation set — in-domain technical evaluation.

In [ ]:
model     = MarianMTModel.from_pretrained("/content/drive/MyDrive/marian_shiksha_final")
tokeniser = MarianTokenizer.from_pretrained("/content/drive/MyDrive/marian_shiksha_final")
model     = model.to(device)

def translate_batch(texts, target_lang=">>mal<<", batch_size=32):
    results = []
    for i in range(0, len(texts), batch_size):
        batch  = [f"{target_lang} {t}" for t in texts[i:i+batch_size]]
        inputs = tokeniser(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=256)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            generated = model.generate(**inputs, num_beams=4, max_length=256)
        decoded = tokeniser.batch_decode(generated, skip_special_tokens=True)
        results.extend(decoded)
    return results

bleu       = BLEU()
sources    = shiksha_val['src_text']
references = shiksha_val['tgt_text']
hypotheses = translate_batch(sources)
score      = bleu.corpus_score(hypotheses, [references])
print(f"Shiksha val BLEU: {score}")

Shiksha val BLEU: BLEU = 32.61 66.6/40.9/26.2/17.4 (BP = 0.976 ratio = 0.976 hyp_len = 218142 ref_len = 223401)


### 6.2 Out-of-Domain Evaluation: arXiv ML Abstracts

We evaluate qualitatively on ML arXiv abstracts — text the model has never seen, from a domain (academic papers) distinct from both BPCC (general) and Shiksha (lectures). No Malayalam references exist for this corpus, so BLEU is not computable. This is intentional: we use arXiv as a probe for generalisation failure, not as a scored benchmark.

**LaTeX handling:** arXiv abstracts contain LaTeX math notation (`$\epsilon$`, `\mathcal{L}`) which the SentencePiece tokeniser has never seen. It segments LaTeX commands into arbitrary subword units, producing phonetic transliterations of symbol names. A cleaning step strips all math notation before translation.

In [ ]:
def clean_abstract(text):
    """Strip LaTeX notation from arXiv abstracts."""
    text = re.sub(r'\$\$.*?\$\$', '', text)
    text = re.sub(r'\$.*?\$', '', text)
    text = re.sub(r'\\[a-zA-Z]+\{.*?\}', '', text)
    text = re.sub(r'\\[a-zA-Z]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

arxiv  = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
sample = arxiv.shuffle(seed=SEED).select(range(500))

clean_texts = []
for row in sample:
    text = clean_abstract(row['title'] + ". " + row['abstract'])
    if len(text) > 50:
        clean_texts.append(text)

print(f"Clean abstracts: {len(clean_texts)}")
print(clean_texts[0][:300])

In [ ]:
arxiv_translations = translate_batch(clean_texts)

results = [{"src": en, "tgt": ml}
           for en, ml in zip(clean_texts, arxiv_translations)]

with open("/content/drive/MyDrive/arxiv_shiksha_translations.json", "w") as f:
    json.dump(results, f, ensure_ascii=False)
print(f"Saved {len(results)} translations")

Saved 500 translations


In [ ]:
with open("/content/drive/MyDrive/arxiv_shiksha_translations.json") as f:
    data = json.load(f)

print(f"Total pairs: {len(data)}")
print("\nFirst pair:")
print("EN:", data[0]['src'][:200])
print("ML:", data[0]['tgt'][:200])

print("\nTechnical pair example:")
technical = [x for x in data if any(t in x['src'].lower()
             for t in ["attention", "gradient", "embedding", "transformer"])]
print("EN:", technical[0]['src'][:200])
print("ML:", technical[0]['tgt'][:200])

Total pairs: 500

First pair:
EN: Epsilon Consistent Mixup: Structural Regularization with an Adaptive Consistency-Interpolation Tradeoff. In this paper we propose -Consistent Mixup (mu). mu is a data-based structural regularization t
ML: എപിസിലോൺ മിക്സഡ് മിക്സഡ് മിക്സേഷൻ: ഒരു അഡാപ്റ്റീവ് സിനിമേറ്റഡ്-II ഇന്റർപോണൻസിറ്റി ട്രേഡുള്ള സ്ട്രക്ച്ചറൽ റെസിസ്റ്റൻഷ്യലൈസേഷൻ.

Technical pair example:
EN: A novel multi-scale loss function for classification problems in machine learning. We introduce two-scale loss functions for use in various gradient descent algorithms applied to classification
ML: മെഷീൻ ലേണിംഗിൽ വർഗ്ഗീകരണ പ്രശ്നങ്ങൾക്കുള്ള ഒരു നോവൽ-സ് ട്രെയിൻ ലോസ് ഫംഗ്ഷൻ. വിവിധ ഗ്രേഡിയന്റ് പാരമ്പര്യ അൽഗോരിത പ്രവർത്തനങ്ങളിൽ
